# Case study STGAN quality-filtered: 8 e 17 maggio 2019 a t+1/t+6

Analisi esclusivamente **post-hoc**. Il notebook usa le decisioni STGAN global top 1% ricalcolate dopo avere escluso i dropout solari regionali isolati e l'ora di recupero. Le label sono già associate all'esatto `(location, timestamp target)` nelle predizioni evaluation-only. Non riaddestra STGAN o SDE-Net.

I due giorni sono i primi due della classifica STGAN pulita sulle sole coordinate diurne PVGIS (`POA > 10 W/m²`): 8 maggio e 17 maggio. Sono candidati anomali non supervisionati, non etichette meteorologiche già validate.

In [ ]:
import importlib, json, os, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'physiq_pv').is_dir():
    for parent in ROOT.parents:
        if (parent / 'physiq_pv').is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.experiments import sde_pipeline as pipe
from physiq_pv.reporting import anomaly_extremes
anomaly_extremes = importlib.reload(anomaly_extremes)

STGAN_SEED = 20
HORIZONS = (1, 6)
EVALUATION_DIR = Path(os.environ.get(
    'STGAN_POSTHOC_ROOT',
    ROOT / 'outputs' / f'sde_stgan_direct_multihorizon_seed{STGAN_SEED}_quality_filtered',
)).resolve()
PREDICTIONS = EVALUATION_DIR / 'predictions.csv'
DAYTIME_THRESHOLD_WM2 = 10.0
OUT_DIR = Path(os.environ.get(
    'STGAN_RARE_EVENTS_OUT_DIR',
    EVALUATION_DIR / 'stgan_may08_may17_t1_t6_pipeline_style',
)).resolve()
FIGURE_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_PEAK_BY_HORIZON = {
    horizon: EVALUATION_DIR / 'posthoc_by_horizon' / f't_plus_{horizon}' / 'reference_production_peaks.csv'
    for horizon in HORIZONS
}
EVENTS = {
    'may_08': {
        'label': '8 maggio 2019',
        'days': ('2019-05-08',),
        'context_start': '2019-05-07',
        'context_end_exclusive': '2019-05-10',
        'selection': 'primo giorno per numero di flag STGAN dopo il filtro qualità',
    },
    'may_17': {
        'label': '17 maggio 2019',
        'days': ('2019-05-17',),
        'context_start': '2019-05-16',
        'context_end_exclusive': '2019-05-19',
        'selection': 'secondo giorno per numero di flag STGAN dopo il filtro qualità',
    },
}
CSV_CHUNKSIZE = 500_000
print('Predizioni   :', PREDICTIONS)
print('Output       :', OUT_DIR)

## 1. Controllo delle sorgenti

Prima eseguire `stgan_pointwise_posthoc_sdenet.ipynb`, che crea le predizioni evaluation-only e i riferimenti di produzione separati per t+1 e t+6.

In [ ]:
required_paths = [
    PREDICTIONS, EVALUATION_DIR / 'evaluation_source.json',
    *REFERENCE_PEAK_BY_HORIZON.values(),
]
missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'File mancanti; eseguire prima stgan_pointwise_posthoc_sdenet.ipynb:\n'
        + '\n'.join(map(str, missing))
    )
metadata = json.loads(
    (EVALUATION_DIR / 'evaluation_source.json').read_text(encoding='utf-8')
)
if metadata.get('detector') != 'stgan':
    raise ValueError(f"Evaluation detector non STGAN: {metadata.get('detector')!r}")
prediction_header = set(pd.read_csv(PREDICTIONS, nrows=0).columns)
required_prediction_columns = {
    'location', 'timestamp', 'horizon_hours', 'anomaly_group', 'y_true',
    'lower_pi', 'upper_pi', 'solar_irradiance_poa_target',
    'detector_anomaly_score', 'detector_is_anomaly',
}
if not required_prediction_columns <= prediction_header:
    raise ValueError(
        f'Colonne predizioni mancanti: {sorted(required_prediction_columns - prediction_header)}'
    )
if not ({'y_pred', 'y_pred_mean'} & prediction_header):
    raise ValueError('Le predizioni richiedono y_pred oppure y_pred_mean.')
if 'event_group' in prediction_header:
    raise ValueError('Le predizioni contengono event_group: non sono STGAN-only.')
if metadata.get('quality_filter_policy') != 'isolated_regional_solar_dropout_plus_immediate_recovery':
    raise ValueError('Le predizioni non dichiarano il filtro qualità STGAN richiesto.')
if not np.isclose(float(metadata.get('clean_top_k_percent', np.nan)), 1.0):
    raise ValueError('Il case study richiede label STGAN clean global top 1%.')
display(pd.DataFrame([metadata]))
print('OK: label STGAN puntuali e forecast direct t+1/t+6 disponibili.')

## 2. Verifica data-driven degli eventi

La classifica usa `detector_is_anomaly` quality-filtered e una sola copia delle coordinate target (`horizon_hours=1`), con filtro puntuale `POA > 10 W/m²`. Il numero di flag misura la persistenza complessiva; la quota massima oraria misura l'estensione spaziale simultanea.

In [ ]:
def _boolean_flags(values):
    flags = values.astype(str).str.strip().str.lower().map(
        {'true': True, 'false': False, '1': True, '0': False}
    )
    if flags.isna().any():
        raise ValueError('detector_is_anomaly contiene valori non booleani.')
    return flags.astype(bool)

selected_dates = tuple(
    day for event in EVENTS.values() for day in event['days']
)
event_by_day = {
    pd.Timestamp(day).normalize(): event['label']
    for event in EVENTS.values() for day in event['days']
}
regional_parts = []
label_parts = []
ranking_columns = [
    'location', 'timestamp', 'horizon_hours', 'solar_irradiance_poa_target',
    'detector_anomaly_score', 'detector_is_anomaly',
]
for chunk in pd.read_csv(
    PREDICTIONS, usecols=ranking_columns, dtype={'location': 'string'},
    chunksize=CSV_CHUNKSIZE,
):
    horizon = pd.to_numeric(chunk['horizon_hours'], errors='coerce')
    poa = pd.to_numeric(chunk['solar_irradiance_poa_target'], errors='coerce')
    selected = horizon.eq(HORIZONS[0]) & poa.gt(DAYTIME_THRESHOLD_WM2)
    chunk = chunk.loc[selected].copy()
    if chunk.empty:
        continue
    chunk['timestamp'] = pd.to_datetime(chunk['timestamp'], errors='raise', utc=True).dt.tz_convert(None)
    chunk['score'] = pd.to_numeric(chunk['detector_anomaly_score'], errors='raise')
    chunk['is_anomaly'] = _boolean_flags(chunk['detector_is_anomaly'])
    grouped = chunk.groupby('timestamp')
    regional_parts.append(pd.DataFrame({
        'n_scored': grouped.size(),
        'n_extreme': grouped['is_anomaly'].sum(),
        'score_max': grouped['score'].max(),
        'score_sum': grouped['score'].sum(),
    }))
    day = chunk['timestamp'].dt.normalize()
    chosen = day.isin(event_by_day) & chunk['is_anomaly']
    if chosen.any():
        label_parts.append(pd.DataFrame({
            'location': chunk.loc[chosen, 'location'].astype(str).to_numpy(),
            'timestamp': chunk.loc[chosen, 'timestamp'].to_numpy(),
            'category': day.loc[chosen].map(event_by_day).to_numpy(),
        }))
if not regional_parts:
    raise ValueError('Nessuna coordinata diurna quality-filtered disponibile.')
regional = pd.concat(regional_parts).groupby(level=0).agg(
    n_scored=('n_scored', 'sum'), n_extreme=('n_extreme', 'sum'),
    score_max=('score_max', 'max'), score_sum=('score_sum', 'sum'),
).sort_index()
regional.index.name = 'timestamp'
regional['score_mean'] = regional['score_sum'] / regional['n_scored']
regional['extreme_share'] = regional['n_extreme'] / regional['n_scored']
regional = regional.drop(columns='score_sum')
daily = anomaly_extremes.rank_flagged_days(regional)
daily.to_csv(OUT_DIR / 'stgan_daily_anomaly_ranking.csv', index=False)
top_days = daily.sort_values(
    ['n_anomalies', 'anomaly_share'], ascending=False
).head(15)
selected_daily = daily[daily['day'].isin(pd.to_datetime(selected_dates))].copy()
display(Markdown('### Quindici giorni STGAN più anomali'))
display(top_days[[
    'day', 'n_anomalies', 'anomaly_share', 'n_active_hours', 'score_max',
]])
display(Markdown('### Giorni scelti per i case study'))
display(selected_daily[[
    'day', 'n_anomalies', 'anomaly_share', 'n_active_hours', 'score_max',
]])

fig, axes = plt.subplots(len(EVENTS), 1, figsize=(14, 7), squeeze=False)
for axis, (event_name, event) in zip(axes.flat, EVENTS.items()):
    start = pd.Timestamp(event['context_start'])
    end = pd.Timestamp(event['context_end_exclusive'])
    frame = regional.loc[(regional.index >= start) & (regional.index < end)]
    axis.plot(frame.index, frame['extreme_share'], color='tab:red', marker='o')
    for day in event['days']:
        axis.axvspan(pd.Timestamp(day), pd.Timestamp(day) + pd.Timedelta(days=1), color='red', alpha=0.10)
    axis.set(title=event['label'], ylabel='Quota località anomale')
    axis.grid(alpha=0.25)
axes[-1, 0].set_xlabel('Timestamp STGAN / target SDE-Net')
fig.suptitle('Estensione regionale delle anomalie STGAN')
fig.tight_layout()
regional_figure = FIGURE_DIR / 'stgan_selected_events_regional_share.png'
fig.savefig(regional_figure, dpi=180, bbox_inches='tight')
plt.show()

## 3. Label puntuali nei soli giorni evento

Le righe sono state selezionate in streaming dalle predizioni quality-filtered. Una previsione entra nella categoria rara soltanto quando la stessa località e lo stesso timestamp target hanno `detector_is_anomaly=True`.

In [ ]:
if not label_parts:
    raise ValueError('Nessun flag STGAN trovato nei giorni selezionati.')
event_labels = pd.concat(label_parts, ignore_index=True)
if event_labels.duplicated(['location', 'timestamp']).any():
    raise ValueError('Label STGAN duplicate per location-timestamp.')
event_labels.to_csv(OUT_DIR / 'stgan_event_pointwise_labels.csv', index=False)
display(event_labels.groupby('category').agg(
    n_flags=('location', 'size'), n_locations=('location', 'nunique'),
))

## 4. Risposta temporale SDE-Net a t+1 e t+6

Ogni pannello usa tutte le località, mostra previsione, target, intervallo predittivo, MAE/RMSE, PICP e incertezza. Le date evidenziate sono timestamp target, non timestamp di emissione.

In [ ]:
diagnostics = {}
diagnostic_summaries = []
for event_name, event in EVENTS.items():
    for horizon_hours in HORIZONS:
        result = pipe.build_extreme_event_diagnostic(
            str(EVALUATION_DIR),
            start=event['context_start'],
            end=event['context_end_exclusive'],
            figure_subdir=f'events/stgan_may08_may17_t1_t6_pipeline_style/{event_name}/t_plus_{horizon_hours}',
            horizon_hours=horizon_hours,
        )
        diagnostics[(event_name, horizon_hours)] = result
        summary = result['summary'].copy()
        summary.insert(0, 'event', event['label'])
        diagnostic_summaries.append(summary)
        display(Markdown(f"### {event['label']} — forecast t+{horizon_hours}"))
        display(summary)
        display(Image(filename=str(result['figure_path'])))
diagnostic_summary = pd.concat(diagnostic_summaries, ignore_index=True)
diagnostic_summary.to_csv(OUT_DIR / 'temporal_diagnostic_summary.csv', index=False)

## 5. Metriche per bin: normale, 8 maggio e 17 maggio

La suite replica `pvgis_sde_pipeline`: per ciascun bin MAE e NMPIL sono boxplot Tukey esatti, mentre RMSE, PICP e CLC sono bar chart; PICP include il target 0,95. Ogni figura confronta `Normale 2019`, `8 maggio 2019` e `17 maggio 2019`, con la numerosità `n` nelle etichette. La procedura viene ripetuta separatamente per t+1 e t+6.

In [ ]:
CATEGORY_LABELS = {
    'normal_2019': 'Normale 2019',
    '2019-05-08': '8 maggio 2019',
    '2019-05-17': '17 maggio 2019',
}
BIN_ORDER = [
    'daytime_0_20_pct', 'daytime_20_40_pct', 'daytime_40_60_pct',
    'daytime_60_80_pct', 'daytime_80_100_pct',
]
BOX_METRICS = [
    ('mae', 'abs_error', 'Absolute error [W]'),
    ('nmpil', 'row_nmpil', 'NMPIL'),
]
BAR_METRICS = [
    ('rmse', 'RMSE [W]', None),
    ('picp', 'PICP', 0.95),
    ('clc', 'CLC', None),
]

def _ordered_category_rows(metrics, band_name):
    rows = (
        metrics.loc[metrics['bin'].eq(band_name)]
        .set_index('category').reindex(CATEGORY_LABELS).reset_index()
    )
    if rows['count'].isna().any():
        missing = rows.loc[rows['count'].isna(), 'category'].tolist()
        raise ValueError(f'Categorie senza dati per {band_name}: {missing}')
    return rows

def _category_ticks(rows):
    return [
        f"{CATEGORY_LABELS[row['category']]}\n(n={int(row['count']):,})"
        for _, row in rows.iterrows()
    ]

def _save_pipeline_boxplot(rows, prefix, metric_name, ylabel, band_name, horizon_hours, figure_dir):
    boxes = [{
        'label': '',
        'mean': float(row[f'{prefix}_mean']),
        'med': float(row[f'{prefix}_median']),
        'q1': float(row[f'{prefix}_q1']),
        'q3': float(row[f'{prefix}_q3']),
        'whislo': float(row[f'{prefix}_whisker_low']),
        'whishi': float(row[f'{prefix}_whisker_high']),
        'fliers': [],
    } for _, row in rows.iterrows()]
    fig, axis = plt.subplots(figsize=(8, 4.5))
    axis.bxp(
        boxes, showfliers=False, showmeans=True,
        meanprops={
            'marker': 'D', 'markerfacecolor': 'red',
            'markeredgecolor': 'red', 'markersize': 5,
        },
    )
    axis.set_xticks(range(1, len(rows) + 1))
    axis.set_xticklabels(_category_ticks(rows), rotation=30, ha='right')
    axis.set(
        title=f'{metric_name.upper()} — {band_name} (all daytime rows) — forecast t+{horizon_hours}h',
        ylabel=ylabel,
    )
    path = figure_dir / f'{metric_name}_{band_name}_boxplot.png'
    fig.savefig(path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    return path

def _save_pipeline_barchart(rows, metric_name, ylabel, target, band_name, horizon_hours, figure_dir):
    values = pd.to_numeric(rows[metric_name], errors='raise').to_numpy(float)
    fig, axis = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(rows))
    axis.bar(x, values, color='steelblue')
    axis.set_xticks(x)
    axis.set_xticklabels(_category_ticks(rows), rotation=30, ha='right')
    if target is not None:
        axis.axhline(target, color='r', linestyle='--', linewidth=1, label=f'target {target:g}')
        axis.legend()
    axis.set(
        title=f'{metric_name.upper()} — {band_name} (all daytime rows) — forecast t+{horizon_hours}h',
        ylabel=ylabel,
    )
    path = figure_dir / f'{metric_name}_{band_name}_bar.png'
    fig.savefig(path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    return path

event_results = {}
event_metric_parts = []
summary_figures = []
for horizon_hours in HORIZONS:
    result = pipe.build_extreme_event_comparison_figures(
        str(EVALUATION_DIR),
        event_dates=selected_dates,
        comparison_name=f'stgan_may08_may17_t_plus_{horizon_hours}',
        horizon_hours=horizon_hours,
        reference_peak_path=REFERENCE_PEAK_BY_HORIZON[horizon_hours],
        generate_figures=False,
    )
    if result['figure_paths']:
        raise RuntimeError('Il calcolo compatto non deve generare figure automatiche.')
    event_results[horizon_hours] = result
    metrics = result['metrics'].copy()
    event_metric_parts.append(metrics)

    horizon_figure_dir = FIGURE_DIR / f't_plus_{horizon_hours}'
    horizon_figure_dir.mkdir(parents=True, exist_ok=True)
    for band_name in BIN_ORDER:
        rows = _ordered_category_rows(metrics, band_name)
        for metric_name, prefix, ylabel in BOX_METRICS:
            figure_path = _save_pipeline_boxplot(
                rows, prefix, metric_name, ylabel, band_name,
                horizon_hours, horizon_figure_dir,
            )
            summary_figures.append(figure_path)
            display(Image(filename=str(figure_path)))
        for metric_name, ylabel, target in BAR_METRICS:
            figure_path = _save_pipeline_barchart(
                rows, metric_name, ylabel, target, band_name,
                horizon_hours, horizon_figure_dir,
            )
            summary_figures.append(figure_path)
            display(Image(filename=str(figure_path)))
    display(Markdown(f'### Dati numerici t+{horizon_hours}'))
    display(metrics[[
        'bin', 'category', 'count', 'mae', 'rmse', 'picp', 'mpiw', 'nmpil', 'clc',
    ]])

event_metrics = pd.concat(event_metric_parts, ignore_index=True)
event_metrics['category_label'] = event_metrics['category'].map(CATEGORY_LABELS)
event_metrics.to_csv(OUT_DIR / 'event_metrics_by_bin_t1_t6.csv', index=False)

## 6. Tracciabilità degli output

In [ ]:
figure_manifest = [regional_figure, *summary_figures] + [
    diagnostics[(event_name, horizon_hours)]['figure_path']
    for event_name in EVENTS for horizon_hours in HORIZONS
]
if len(figure_manifest) != 55:
    raise RuntimeError(f'Attese 55 figure, trovate {len(figure_manifest)}.')
pd.DataFrame({'figure_path': [str(path) for path in figure_manifest]}).to_csv(
    OUT_DIR / 'figure_manifest.csv', index=False,
)
display(pd.DataFrame({'figure_path': [str(path) for path in figure_manifest]}))

analysis_metadata = {
    'analysis': 'stgan_may08_may17_quality_filtered_t1_t6_pipeline_style',
    'post_processing_only': True,
    'training_rerun': False,
    'detector': 'stgan',
    'label_source': 'quality_filtered_clean_global_top_1_percent',
    'quality_filter_policy': metadata['quality_filter_policy'],
    'excluded_quality_timestamps': metadata['data_quality_timestamps'],
    'join_key': ['location', 'timestamp'],
    'timestamps_are_target_times': True,
    'horizons_hours': list(HORIZONS),
    'events': EVENTS,
    'predictions': str(PREDICTIONS),
    'saved_figure_count': len(figure_manifest),
    'metrics': ['mae', 'rmse', 'picp', 'mpiw', 'nmpil', 'clc'],
}
(OUT_DIR / 'analysis_metadata.json').write_text(
    json.dumps(analysis_metadata, indent=2), encoding='utf-8'
)
print('Output principali:')
for path in sorted(OUT_DIR.iterdir()):
    print(' -', path.name)

## Interpretazione

- Il confronto per bin misura l'intero 8 maggio e l'intero 17 maggio, comprese le località non flaggate, rispetto alle coordinate normali del resto del 2019.
- MAE e RMSE misurano l'errore; PICP misura la copertura; NMPIL la larghezza normalizzata dell'intervallo; CLC combina ampiezza e copertura. MPIW e `count` restano nel CSV.
- Il confronto `t+1`/`t+6` mostra se l'evento diventa più difficile aumentando l'orizzonte.
- L'8 e il 17 maggio sono rispettivamente il primo e il secondo giorno della classifica STGAN dopo il filtro qualità.
- Le vecchie date 12 giugno e 2–3 luglio non sono usate come eventi fisici: il loro primato originario dipendeva dai dropout regionali e dal recupero immediato.

STGAN resta un detector non supervisionato: `raro` non significa automaticamente evento meteorologico reale né forecast difficile.

## Risultati da copiare

Eseguire questa cella per ottenere i blocchi CSV da inviare per l'analisi. Include metriche per bin, posizione dei due eventi nella classifica STGAN pulita e riepilogo temporale.

In [ ]:
copy_metrics_columns = [
    'horizon_hours', 'bin', 'category', 'category_label', 'count',
    'mae', 'rmse', 'picp', 'mpiw', 'nmpil', 'clc',
    'abs_error_mean', 'abs_error_q1', 'abs_error_median', 'abs_error_q3',
    'abs_error_whisker_low', 'abs_error_whisker_high',
    'row_nmpil_mean', 'row_nmpil_q1', 'row_nmpil_median', 'row_nmpil_q3',
    'row_nmpil_whisker_low', 'row_nmpil_whisker_high',
]
copy_metrics = event_metrics[copy_metrics_columns].sort_values(
    ['horizon_hours', 'bin', 'category']
).reset_index(drop=True)
copy_metrics_path = OUT_DIR / 'stgan_event_bin_metrics_copy_report.csv'
copy_metrics.to_csv(copy_metrics_path, index=False)

ranked_daily = daily.sort_values(
    ['n_anomalies', 'anomaly_share'], ascending=False
).reset_index(drop=True)
ranked_daily.insert(0, 'clean_rank', np.arange(1, len(ranked_daily) + 1))
copy_ranking = ranked_daily[
    ranked_daily['day'].isin(pd.to_datetime(selected_dates))
].copy()
copy_ranking['day'] = pd.to_datetime(copy_ranking['day']).dt.strftime('%Y-%m-%d')
copy_ranking_path = OUT_DIR / 'stgan_selected_days_copy_report.csv'
copy_ranking.to_csv(copy_ranking_path, index=False)

copy_diagnostics = diagnostic_summary.copy()
copy_diagnostics_path = OUT_DIR / 'stgan_temporal_diagnostics_copy_report.csv'
copy_diagnostics.to_csv(copy_diagnostics_path, index=False)

print('BEGIN_STGAN_EVENT_BIN_METRICS_CSV')
print(copy_metrics.to_csv(index=False, float_format='%.6g').strip())
print('END_STGAN_EVENT_BIN_METRICS_CSV')
print('BEGIN_STGAN_SELECTED_DAYS_CSV')
print(copy_ranking.to_csv(index=False, float_format='%.6g').strip())
print('END_STGAN_SELECTED_DAYS_CSV')
print('BEGIN_STGAN_TEMPORAL_DIAGNOSTICS_CSV')
print(copy_diagnostics.to_csv(index=False, float_format='%.6g').strip())
print('END_STGAN_TEMPORAL_DIAGNOSTICS_CSV')
print('File salvati in:', OUT_DIR)